In [ ]:
import pathlib as pl
from pprint import pprint
from shutil import rmtree

import jupyter_black
import numpy as np
import pywatershed as pws
from tqdm import tqdm
# import xarray as xr

jupyter_black.load()  # auto-format the code in this notebook

In [ ]:
domain_dir = pws.constants.__pywatershed_root__ / "data/drb_2yr"
nb_output_dir = pl.Path("./02_prms_legacy_models_monthly_output")

In [ ]:
params = pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")

In [ ]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

In [ ]:
control = pws.Control.load_prms(
    domain_dir / "nhm.control", warn_unused_options=False
)

In [ ]:
cbh_nc_dir = domain_dir
control.options["netcdf_output_var_names"] += ["infil_hru", "sroff_vol"]
control.edit_end_time(np.datetime64("1979-07-01T00:00:00"))
run_dir = nb_output_dir / "nhm"
if run_dir.exists():
    rmtree(run_dir)
control.options = control.options | {
    "input_dir": cbh_nc_dir,
    "budget_type": "warn",
    "calc_method": "numba",
    # "netcdf_output_dir": , # this can be on or off.
}

In [ ]:
nhm = pws.Model(
    nhm_processes,
    control=control,
    parameters=params,
)

In [ ]:
# The poi_gage_segment is an index, but it really should be the portable nhm_seg id.
# Solve for nhm_seg and make a crosswalks both ways, cause why not.
nhm_seg = params.parameters["nhm_seg"]
poi_gage_segment = params.parameters["poi_gage_segment"]
poi_nhm_seg = nhm_seg[poi_gage_segment - 1]  # fortran indexing
poi_id_nhm_seg = dict(
    zip(params.parameters["poi_gage_id"], poi_nhm_seg.tolist())
)
poi_nhm_seg_id = dict(
    zip(poi_nhm_seg.tolist(), params.parameters["poi_gage_id"])
)

In [ ]:
def max(da, dim=None, *, skipna=None, keep_attrs=None, **kwargs):
    return da.max(dim=dim, skipna=skipna, keep_attrs=keep_attrs, **kwargs)


output = pws.base.CustomOutput(
    control=control,
    model=nhm,
    monthly_accum_var_list=[
        "sroff",
        "hru_actet",
    ],
    # monthly_accum_stats=["accum", "mean"],
    poi_var_list=[
        "seg_outflow",
    ],
    poi_nhm_seg=poi_nhm_seg,
    poi_gage_segment=poi_gage_segment - 1,
    poi_stats=[
        "mean",
        "median",
        max,
    ],
    poi_stats_groupby={"median": "month"},
    poi_stats_resample={"median": "1MS", "max": "5d"},
    hru_sub_var_list=[
        "hru_actet",
        "pkwater_equiv",
    ],
    hru_sub_ids=[params.parameters["nhm_id"][0].tolist()],
    hru_sub_stats=["mean", max],
    hru_sub_stats_resample={"mean": "1MS", "max": "1YS"},
)

In [ ]:
for tt in tqdm(range(control.n_times)):
    nhm.advance()
    nhm.calculate()
    output.calculate()

nhm.finalize()
output.finalize()

In [ ]:
control.itime_step

In [ ]:
output.monthly_accumulations

In [ ]:
output.n_days_per_month

In [ ]:
output.poi_arrays

In [ ]:
assert (
    nhm.processes["PRMSChannel"]["seg_outflow"][poi_gage_segment - 1]
    == output.poi_arrays["seg_outflow"][-1, :]
).all()

In [ ]:
pprint(output.poi_stats)

In [ ]:
pprint(output.hru_sub_stats)